# Build Graph using Cognee (default configuration)

This notebook demonstrates how to use Cognee to ingest markdown files and query the database through a large language model.

It covers the following steps:
1. Ingesting the markdown files into Cognee
2. Check the data ingestion quality
3. Generate the graph of ingested data in an interactive visualization
4. Query the data by asking a question (e.g., "What is the difference between Bitcoin and Ethereum in terms of their technology, use cases, and consensus mechanisms?")
5. Analyze the evidence provided by the large language model in response to the query. 
6. Add used nodes to an interactive graph to illustrate the path taken by the model to provide its conclusions.

The following notebook is executed under the following configuration file:
```
LLM_PROVIDER="custom"
LLM_MODEL="openrouter/openai/gpt-5-nano"
LLM_ENDPOINT="https://openrouter.ai/api/v1"
LLM_API_KEY="sk-or-v1-7ea...d"
EMBEDDING_PROVIDER="custom"
EMBEDDING_MODEL="openrouter/openai/text-embedding-3-small"
EMBEDDING_API_KEY="sk-or-v1-7ea...d"
SYSTEM_ROOT_DIRECTORY="D:/cognee/system"
DATA_ROOT_DIRECTORY="D:/cognee/data"
```

In [ ]:
from pathlib import Path
from time import perf_counter

import cognee

# Delete existing data
await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
await cognee.forget(everything=True)

# Data
data_path = "../../data/silver"

files = sorted(
    path for path in Path(data_path).rglob("*.md")
    if path.is_file() and path.stat().st_size > 0
)
for path in files:
    print(f"  - {path.relative_to(data_path)}")

# Ingestion
start = perf_counter()
await cognee.remember(
[str(path) for path in files],
dataset_name="docspace",
self_improvement=False,
)
print(f"Ingestion completed {(perf_counter() - start) / 60:.1f} min")


2026-11-13T19:04:58.730051 [info     ] Log file created at: C:\Users\moote\.cognee\logs\2026-11-13_19-04-56.log [cognee.shared.logging_utils] log_file=C:\Users\moote\.cognee\logs\2026-11-13_19-04-56.log

2026-11-13T19:04:58.733055 [warning  ] Cognee 1.0 changes: New API — remember/recall/forget/improve (V1 add/cognify/search still work). Session memory enabled by default (CACHING=false to disable). Multi-user access control on by default (ENABLE_BACKEND_ACCESS_CONTROL=false to disable). Agents (@cognee.agent) auto-verified on registration. See https://docs.cognee.ai/ [cognee.shared.logging_utils]

2026-11-13T19:04:58.734049 [info     ] Logging initialized            [cognee.shared.logging_utils] cognee_version=1.6.7 database_path=D:\cognee\system\databases os_info='Windows 10 (10.0.26300)' python_version=3.11.6 structlog_version=25.5.0

2026-11-13T19:04:58.737475 [info     ] Database storage: D:\cognee\system\databases [cognee.shared.logging_utils]

2026-11-13T19:04:58.739484 [info   

  - bitcoin\A Peer-to-Peer Electronic Cash System.md
  - bitcoin\Fact Sheet iShares.md
  - cyber\Exploring Cryptocurrency.md
  - defi\BIS bulletin-76.md
  - reports\21Shares Report June 2025.md
  - reports\crypto_global\weekly\CryptoGlobal_Weekly_2026-07-11.md
  - reports\crypto_global\weekly\CryptoGlobal_Weekly_2026-07-13.md
  - reports\imf\Crypto Assets Monitor.md
  - reports\imf\Crypto Monitor Q1 2026.md
  - reports\imf\GMM Special Feature - Crypto Monitor October 2025.md
  - reports\imf\Workshop on Fraud and Investigations Involving Crypto Assets.md
  - stablecoins\Brief remarks on stablecoins.md
  - stablecoins\bulletin-108-stablecoin-growth-policy-challenges-and-approaches.md
  - stablecoins\bulletin-125-stablecoin-remuneration-centralised-exchanges.md
Ingestion completed 0.0 min


# Ingestion Checks

In [2]:
from collections import Counter, defaultdict

from cognee.context_global_variables import set_database_global_context_variables
from cognee.infrastructure.databases.graph import get_graph_engine

# 1. Get ingested documents
datasets = await cognee.datasets.list_datasets()
dataset = next((d for d in datasets if d.name == "docspace"), None)

if dataset is None:
    raise ValueError("Dataset not found: docspace")

documents = await cognee.datasets.list_data(dataset.id)

# 2. Get graph data
async with set_database_global_context_variables(
    dataset.id, dataset.owner_id
):
    graph = await get_graph_engine()
    nodes, edges = await graph.get_graph_data()

node_by_id = {
    node_id: props
    for node_id, props in nodes
}

types = Counter(
    props.get("type", "Unknown")
    for _, props in nodes
)

graph_docs = {
    node_id: props
    for node_id, props in node_by_id.items()
    if props.get("type") in ("TextDocument", "Document")
}

chunks = {
    node_id: props
    for node_id, props in node_by_id.items()
    if props.get("type") == "DocumentChunk"
}

chunks_by_doc = defaultdict(set)
parents_by_chunk = defaultdict(set)

for source, target, relation, props in edges:
    source, target = source, target

    if source in graph_docs and target in chunks:
        chunks_by_doc[source].add(target)
        parents_by_chunk[target].add(source)

    elif target in graph_docs and source in chunks:
        chunks_by_doc[target].add(source)
        parents_by_chunk[source].add(target)

# Detect chunks without source document
orphan_chunks = [
    cid for cid in chunks
    if not parents_by_chunk[cid]
]

# Detect chunks with multiple source documents
multi_parent_chunks = [
    cid for cid, parents in parents_by_chunk.items()
    if len(parents) > 1
]

# 3. Summary
print("=" * 70)
print("Summary for dataset: docspace")
print("=" * 70)

print(f"Local files       : {len(files)}")
print(f"Ingested documents : {len(documents)}")
print(f"Graph documents   : {len(graph_docs)}")
print(f"Graph chunks      : {len(chunks)}")
print(f"Orphan chunks    : {len(orphan_chunks)}")
print(f"Multi-parent chunks: {len(multi_parent_chunks)}")
print(f"Total nodes          : {len(nodes)}")
print(f"Total edges     : {len(edges)}")

print("\nINGESTED DOCUMENTS")
for doc in documents:
    print(f"- {doc.name} | {doc.id}")

print("\nCHUNKS PER GRAPH DOCUMENT")
for doc_id, props in sorted(
    graph_docs.items(),
    key=lambda x: str(x[1].get("name", ""))
):
    name = props.get("name", doc_id)
    count = len(chunks_by_doc[doc_id])
    print(f"- {name}: {count} chunks")

print("\nNODE TYPES")
for node_type, count in types.most_common():
    print(f"- {node_type}: {count}")

print("\nCHECKS")
if files and len(files) != len(documents):
    print("[WARNING] Number of local files differs from dataset")

if len(documents) != len(graph_docs):
    print("[INFO] Number of documents differs between dataset and graph")

if orphan_chunks:
    print(f"[WARNING] {len(orphan_chunks)} chunks without direct link to a document")

if (
    (not files or len(files) == len(documents))
    and len(documents) == len(graph_docs)
    and not orphan_chunks
):
    print("[OK] No discrepancies detected")


2026-11-13T19:07:17.066291 [info     ] Retrieved 415 nodes and 933 edges in 0.10 seconds [cognee.shared.logging_utils]


Summary for dataset: docspace
Local files       : 14
Ingested documents : 14
Graph documents   : 14
Graph chunks      : 52
Orphan chunks    : 0
Multi-parent chunks: 0
Total nodes          : 415
Total edges     : 933

INGESTED DOCUMENTS
- BIS bulletin-76 | 31fefbb7-0cf3-43f7-af84-a57f0aa0f47d
- Crypto Assets Monitor | 18a2bb05-0aac-41f7-8272-11e7acf246e4
- Brief remarks on stablecoins | 76b6d6f0-f0c0-4450-9bda-d3bdb63af4be
- Crypto Monitor Q1 2026 | eff6b115-aaee-4e0a-9528-549fab82d947
- 21Shares Report June 2025 | cfbda14e-7d76-4eb1-88ef-62250fe2b70b
- Fact Sheet iShares | 0bc16987-5069-44a4-a25a-42a503fd64ad
- Workshop on Fraud and Investigations Involving Crypto Assets | 1319227e-54ee-4536-b43e-e1c03894ecd7
- GMM Special Feature - Crypto Monitor October 2025 | 5f58b1f3-c2ae-4e4c-a047-19f867ee1ada
- CryptoGlobal_Weekly_2026-07-13 | 8fa495d3-c584-432e-9331-c7064b5add59
- bulletin-125-stablecoin-remuneration-centralised-exchanges | 8678a59b-0689-4dc4-abed-39bf5718dff2
- CryptoGlobal_Wee

# Graph Visualization (Interactive HTML)

In [3]:
import webbrowser

await cognee.visualize_graph("graph.html", dataset="docspace", full=True)
webbrowser.open("graph.html")


2026-11-13T19:07:23.219077 [info     ] Retrieved 415 nodes and 933 edges in 0.04 seconds [cognee.shared.logging_utils]

2026-11-13T19:07:23.351262 [info     ] Collected 14 session operation events for the Memory tab. [visualization.session_events]

2026-11-13T19:07:59.048876 [info     ] fetch_node_embeddings: resolved 337/415 node embeddings across 5 collection(s) [embedding_join]

2026-11-13T19:08:02.557008 [info     ] Graph visualization saved as graph.html [cognee.shared.logging_utils]

2026-11-13T19:08:02.561005 [info     ] The HTML file has been stored at path: graph.html [cognee.shared.logging_utils]


True

# Queryin the graph for cryptocurrency information (Search type: GRAPH_COMPLETION)

In [4]:
results = await cognee.recall(
    query_text = "What is the difference between Bitcoin and Ethereum in terms of their technology, use cases, and consensus mechanisms?",
    query_type=cognee.SearchType.GRAPH_COMPLETION,
    top_k=15,
    system_prompt="You are a knowledgeable assistant on cryptocurrencies. Provide detailed and accurate responses.",
    include_references=True,
    # only_context=True
)
for result in results:
    print("=" * 70)
    print(f"Result: {result.text}")
    print(f"Dataset used: {result.dataset_name}")
    evidence = result.metadata.get("evidence", [])


2026-11-13T19:08:11.257503 [info     ] Vector collection retrieval completed: Retrieved distances from 7 collections in 0.48s [cognee.shared.logging_utils]

2026-11-13T19:08:11.259909 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-11-13T19:08:11.463199 [info     ] ID-filtered retrieval: 415 nodes and 878 edges in 0.20s [cognee.shared.logging_utils]

2026-11-13T19:08:11.510363 [info     ] Graph projection completed: 415 nodes, 878 edges in 0.04s [CogneeGraph]

2026-11-13T19:08:11.558689 [warning  ] No nodes found in the database [cognee.shared.logging_utils]

2026-11-13T19:08:11.562357 [info     ] Vector collection retrieval completed: Retrieved distances from 7 collections in 0.78s [cognee.shared.logging_utils]

2026-11-13T19:08:11.565889 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-11-13T19:08:11.693215 [info     ] ID-filtered retrieval: 415 nodes and 853 edges in 0.12s [cognee.shared.logging_utils]

2026-11-13T19:08:11.7499

Result: Here’s a concise three-section comparison:

Technology
- Bitcoin
  - UTXO-based ledger for value transfer.
  - Limited scripting; designed for simple, secure payments.
  - Blockchain secured by Proof-of-Work (PoW) with SHA-256; Merkle trees for transaction hashing; timestamped, append-only chain.
- Ethereum
  - Account-based (stateful) ledger with the Ethereum Virtual Machine (EVM) for smart contracts.
  - Turing-complete programming environment; supports DeFi, NFTs, etc.
  - Gas mechanism to price computation/storage; post-Merge PoS consensus.

Use cases
- Bitcoin
  - Digital cash and store of value; peer-to-peer value transfer and settlement.
  - Limited on-chain programmability.
- Ethereum
  - Programmable money and decentralized applications (DeFi, NFTs, DAOs, tokenized services).
  - Rich ecosystem of smart contracts and token standards (ERC-20, ERC-721, etc.).

Consensus mechanisms
- Bitcoin
  - Proof-of-Work (PoW); the longest chain with the most accumulated work is cano

## Evidence Analysis

In [5]:
def get_node_infos(node_id):
    node_props = node_by_id.get(node_id, {})
    node_type = node_props.get("type", "Unknown")

    # Entity node
    if node_type=="Entity":
        name = node_props.get("name")
        description = node_props.get("description")
        return {
            "id": node_id,
            "type": node_type,
            "name": name,
            "description": description
        }

    # TextSummary node
    elif node_type=="TextSummary":
        text = node_props.get("text")
        source_chunk_id = node_props.get("source_chunk_id")
        if source_chunk_id:
            source_chunk_props = get_node_infos(source_chunk_id)
            source_chunk_doc = source_chunk_props.get("document_name")
        return {
            "id": node_id,
            "type": node_type,
            "text": text[:100],
            "source_chunk_id": source_chunk_id,
            "source_chunk_doc": source_chunk_doc if source_chunk_id else None
        }

    # Timestamp node
    elif node_type=="Timestamp":
        timestamp = node_props.get("timestamp_str")
        return {
            "id": node_id,
            "type": node_type,
            "timestamp": timestamp
        }

    # EntityType node
    elif node_type=="EntityType":
        description = node_props.get("description")
        return {
            "id": node_id,
            "type": node_type,
            "description": description
        }

    # Chunk node
    elif node_type=="DocumentChunk":
        text = node_props.get("text")
        chunk_size = node_props.get("chunk_size")
        chunk_index = node_props.get("chunk_index")
        cut_type = node_props.get("cut_type")
        document_id = node_props.get("document_id")
        document_name = node_props.get("document_name")
        return {
            "id": node_id,
            "type": node_type,
            "text": text[:100],
            "chunk_size": chunk_size,
            "chunk_index": chunk_index,
            "cut_type": cut_type,
            "document_id": document_id,
            "document_name": document_name
        }

    # Document node
    elif node_type=="TextDocument":
        document_name = node_props.get("name")
        return {
            "id": node_id,
            "type": node_type,
            "document_name": document_name
        }

    # Unknown node type
    else:
        return {
            "id": node_id,
            "type": node_type,
            "description": "WARNING UNKNOWN NODE TYPE!"
        }

evidence = results[0].metadata["evidence"] # Extract the evidence from the first result
# Iterate through the evidence and display its details
for i, ev in enumerate(evidence, 1):
    kind = ev.get("kind")
    artifact_id = ev.get("artifact_id")
    role = ev.get("role")
    rank = ev.get("rank")

    print(f"\n{'─' * 90}")
    print(f"[{i}] {kind.upper()} | Rank: {rank} | Role: {role}")
    print(f"{'─' * 90}")

    if kind == "graph_node":
        node_props = get_node_infos(artifact_id)
        for key, value in node_props.items():
            print(f"{key}: {value}")

    elif kind == "graph_edge":
        source_id, target_id = ev.get("source_node_id"), ev.get("target_node_id")
        relation = ev.get("relationship_name")
        print(f"Source: {source_id} -> Target: {target_id} | Relation: {relation}")
        source_props = get_node_infos(source_id)
        target_props = get_node_infos(target_id)
        print("Source node properties:")
        for key, value in source_props.items():
            print(f"  {key}: {value}")
        print("Target node properties:")
        for key, value in target_props.items():
            print(f"  {key}: {value}")

    elif kind == "segment":

        # Chunk
        chunk_id = ev.get("chunk_id")
        if chunk_id:
            chunk_props = get_node_infos(chunk_id)
            print("Chunk properties:")
            for key, value in chunk_props.items():
                print(f"  {key}: {value}")

        # Relation
        source_id, target_id = ev.get("source_node_id"), ev.get("target_node_id")
        relation = ev.get("relationship_name")
        if source_id and target_id and relation:
            print(f"Source: {source_id} -> Target: {target_id} | Relation: {relation}")
            source_props = get_node_infos(source_id)
            target_props = get_node_infos(target_id)
            print("Source node properties:")
            for key, value in source_props.items():
                print(f"  {key}: {value}")
            print("Target node properties:")
            for key, value in target_props.items():
                print(f"  {key}: {value}")

    else:
        print(f"Unknown evidence kind: {kind}")


──────────────────────────────────────────────────────────────────────────────────────────
[1] GRAPH_NODE | Rank: 0 | Role: used_as_context
──────────────────────────────────────────────────────────────────────────────────────────
id: 424d8229-49fb-548d-874e-579265aaec59
type: TextSummary
text: person: satoshi nakamoto, adam back
concept: bitcoin, merkle tree, block, electronic coin, bitcoin n
source_chunk_id: 95e50490-92a5-59f0-bb9a-8c273d573165
source_chunk_doc: A Peer-to-Peer Electronic Cash System

──────────────────────────────────────────────────────────────────────────────────────────
[2] GRAPH_NODE | Rank: 1 | Role: used_as_context
──────────────────────────────────────────────────────────────────────────────────────────
id: 95e50490-92a5-59f0-bb9a-8c273d573165
type: DocumentChunk
text: # Bitcoin: A Peer-to-Peer Electronic Cash System

Satoshi Nakamoto [  
satoshin@gmx.com](mailto:sato
chunk_size: 4010
chunk_index: 0
cut_type: paragraph_end
document_id: a1af1bd1-8661-4ac6-8a42

In [ ]:
from notebooks.cognee.highlight_decision import highlight_cognee_html

highlight_cognee_html(
    evidence=evidence,
    source="graph.html",
    output="graph_highlighted.html",
)


2026-11-13T19:19:18.307683 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]
